# 📝 Dockerfile for LLM Apps

**Write production-ready Dockerfiles**

## 📋 Overview

**What you'll learn:**
- Dockerfile best practices
- Multi-stage builds
- Layer caching
- Security considerations

**Time estimate:** ⏱️ 50 minutes

## 📝 Basic Dockerfile

```dockerfile
# Dockerfile for FastAPI LLM app
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Copy requirements
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose port
EXPOSE 8000

# Run application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

## 🏗️ Multi-Stage Build

```dockerfile
# Stage 1: Builder
FROM python:3.11-slim AS builder

WORKDIR /app

# Install build dependencies
RUN apt-get update && apt-get install -y \
    gcc \
    g++ \
    && rm -rf /var/lib/apt/lists/*

# Install Python packages
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: Runtime
FROM python:3.11-slim

WORKDIR /app

# Copy only installed packages from builder
COPY --from=builder /root/.local /root/.local

# Make sure scripts are in PATH
ENV PATH=/root/.local/bin:$PATH

# Copy application
COPY . .

# Non-root user for security
RUN useradd -m appuser && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Benefits:**
- ✅ Smaller final image (no build tools)
- ✅ Faster builds (cached layers)
- ✅ More secure (no compilers in prod)

## 🚀 Production Dockerfile

```dockerfile
FROM python:3.11-slim AS base

# Install system dependencies
RUN apt-get update && apt-get install -y \
    curl \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Builder stage
FROM base AS builder

COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Runtime stage
FROM base AS runtime

# Copy dependencies
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

# Copy application
COPY --chown=appuser:appuser . .

# Create non-root user
RUN useradd -m appuser
USER appuser

# Health check
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
```

## ✅ Summary

**Dockerfile best practices:**

1. **Use specific base images**
```dockerfile
# ❌ Bad
FROM python

# ✅ Good
FROM python:3.11-slim
```

2. **Order layers by change frequency**
```dockerfile
# Least frequently changed first
COPY requirements.txt .
RUN pip install -r requirements.txt

# Most frequently changed last
COPY . .
```

3. **Minimize layers**
```dockerfile
# ❌ Bad - 3 layers
RUN apt-get update
RUN apt-get install -y curl
RUN rm -rf /var/lib/apt/lists/*

# ✅ Good - 1 layer
RUN apt-get update && apt-get install -y curl \
    && rm -rf /var/lib/apt/lists/*
```

4. **Use .dockerignore**
```
__pycache__
*.pyc
.git
.env
node_modules
```

### Next: `13_docker/03_docker_compose.ipynb`